# Figures of NGC 6383's paper by Pulgar-Escobar et al. 2024.

## Module import

In [ ]:
from COSMIC_aux import *

In [ ]:
from astropy.visualization import quantity_support
from astropy.table import QTable, join
from astropy.io import ascii, fits
from astropy.wcs import WCS
from astropy.stats import sigma_clip
import asteca
from sklearn.utils.extmath import weighted_mode
from astropy.stats import knuth_bin_width
from scipy import stats as stats_sc
from datetime import datetime
from astropy.stats import histogram as ashist
%config InlineBackend.figure_format ='retina'
quantity_support();

In [ ]:
## Read of the data and labelling
data_gaia = QTable.read('40_arcmin_clustered.ecsv', guess=False, format='ascii.ecsv')

In [ ]:
# Read the FITS file
with fits.open('../NGC6383_DSS2-red.fits') as hdulist:
    # Extract the data and header information
    data = hdulist[0].data
    header = hdulist[0].header
wcs = WCS(header);

In [ ]:
cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.5)
noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))

In [ ]:
for i in np.unique(data_gaia['cluster']):
    print(f"There's {len(data_gaia[data_gaia['cluster'] == i])} sources in the cluster {i}")

## Parallax clipping

In [ ]:
print(rf'{len(data_gaia[cluster])} before $\sigma$ clipping')
parmax, min_plx, max_plx = sigma_clip(data_gaia['parallax'][cluster], sigma=2, cenfunc=histogram_mode, stdfunc='std',return_bounds=True)
data_gaia['cluster'][(data_gaia['cluster'] == 0) & ((data_gaia['parallax'] < min_plx) | (data_gaia['parallax'] > max_plx))] = -1
cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.5)
noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))
print(rf'{len(data_gaia[cluster])} after $\sigma$ clipping')

In [ ]:
min_plx, max_plx

In [ ]:
fig_prob, ax_prob = plt.subplots(1,1,figsize=(7,6),layout='tight')
sc_cmap = ax_prob.scatter(data_gaia['Gmag'][cluster],data_gaia['probability'][cluster],s=9,label='NGC 6383 potencial members',c=data_gaia[cluster]['fidelity_v2'],cmap='coolwarm')
fig_prob.colorbar(sc_cmap,pad=0.01).set_label('Astrometric fidelity',fontsize=15)
#ax_prob.scatter(data_gaia['Gmag'][noise],data_gaia['probability'][noise],s=4,label='Field stars',c='blue')
ax_prob.axhline(0.6,color='k',ls='--',label=r'$60\%$ of probability')
ax_prob.set_xlabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit),fontsize=16)
ax_prob.set_ylabel(r'Probability',fontsize=16)
ax_prob.legend(loc='center left')
ax_prob.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_prob.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_prob.tick_params(axis='both', which='both', direction='in')
fig_prob.savefig('../Tex_File/Figures/probabilies_post_sigmaclip.pdf',dpi='figure',bbox_inches='tight');

In [ ]:
prob_thresholds = [0.5, 0.6, 0.7, 0.8]

for i in prob_thresholds:
    condition = data_gaia[cluster]['probability'] >= i 
    condition_mag = (data_gaia[cluster]['probability'] >= i) & (data_gaia[cluster]['Gmag'] <= 19*u.mag)
    print(len(data_gaia[cluster][condition]),len(data_gaia[cluster][condition_mag]))

## CMD

In [ ]:
### sagitta_data = data_gaia[cluster]['source_id','parallax','l','b','Gmag','G_BPmag','G_RPmag','j_m','h_m','ks_m','parallax_error','e_Gmag',
#            'e_G_BPmag','e_G_RPmag','j_msigcom','h_msigcom','ks_msigcom']
#sagitta_data.rename_columns(sagitta_data.colnames,['source_id','parallax','l','b','g','bp','rp','j','h','k','eparallax','eg','ebp','erp','ej','eh','ek'])
#sagitta_data.write('Sagitta/NGC_6383_sagitta.fits',overwrite=True,format='fits')
#!sagitta Sagitta/NGC_6383_sagitta.fits --av_out av_sagitta --pms_out pms_sagitta

In [ ]:
pms = QTable.read('NGC_6383_sagitta-sagitta.fits')
for i in ['av_sagitta','pms_sagitta','age']:
    pms[i] = np.squeeze(pms[i])
if not any(column in data_gaia.colnames for column in ['av_sagitta', 'pms_sagitta', 'age']):
    # If none of the columns exist in data_gaia, perform the join operation
    data_gaia = join(data_gaia, pms['source_id', 'av_sagitta', 'pms_sagitta', 'age'], keys='source_id', join_type='left')
    data_gaia['pms_sagitta'] = data_gaia['pms_sagitta'].filled(0)
    cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.5)
    noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))

In [ ]:
# Example of initial condition you provided
low = cluster & (data_gaia['probability'] < 0.6)
high = cluster & (data_gaia['probability'] >= 0.6)

# Adding conditions for pms_high and pms_low with non-NaN 'tmass_oid'
pms_high = high & (data_gaia['pms_sagitta'] >= 0.6) & ~np.isnan(data_gaia['tmass_oid'])
pms_low = low & (data_gaia['pms_sagitta'] >= 0.6) & ~np.isnan(data_gaia['tmass_oid'])

# Conditions for ms_high and ms_low
ms_high = high & (data_gaia['pms_sagitta'] < 0.6) & (data_gaia['pms_sagitta'] >= 0) & ~np.isnan(data_gaia['tmass_oid'])
ms_low = low & (data_gaia['pms_sagitta'] < 0.6) & (data_gaia['pms_sagitta'] >= 0) & ~np.isnan(data_gaia['tmass_oid'])

# Adding categories for high and low where 'tmass_oid' is NaN
high_tmass_nan = high & np.isnan(data_gaia['tmass_oid'])
low_tmass_nan = low & np.isnan(data_gaia['tmass_oid'])
HD159176 = data_gaia[data_gaia['Gmag']<6*u.mag]

In [ ]:
fig, ax = plt.subplots(3,1,layout='tight',figsize=(7,7))
ax[0].hist(data_gaia[pms_high]['pms_sagitta'],bins='auto', color='orange', histtype='step',hatch='//')
ax[1].hist(data_gaia[pms_high]['age'],bins='auto', color='orange', histtype='step',hatch='//')
ax[2].hist(data_gaia[pms_high]['av_sagitta'],bins='auto', color='orange', histtype='step',hatch='//')
ax[0].set_xlabel('PMS Probability',fontsize=16)
ax[1].set_xlabel('log(Age)',fontsize=16)
ax[2].set_xlabel(r'$A_V$',fontsize=16)
#sc = ax.scatter(data_gaia[cluster]['pms_sagitta'],data_gaia[cluster]['age'],c=data_gaia[cluster]['av_sagitta'],cmap='coolwarm',label='NGC 6383 potencial members',s=12)
for i in ax.flatten():
    i.set_ylabel('Count',fontsize=16)
    i.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    i.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    i.tick_params(axis='both', which='both', direction='in')
fig.savefig('../Tex_File/Figures/pms_stats.pdf',dpi='figure',bbox_inches='tight');

## ASteCA

In [ ]:
isochs = asteca.isochrones(isochs_path="../MIST/",
                           magnitude={"Gaia_G_EDR3": 6390.7},
                           color={"Gaia_BP_EDR3": 5182.6, "Gaia_RP_EDR3": 7825.1},
                           color2 = {"Gaia_RP_EDR3": 7825.1, "2MASS_J": 12375.60}
                          )
#isochs = asteca.isochrones(isochs_path="../PARSEC/",magnitude={"Gmag": 6390.21},color={"G_BPmag": 5182.58, "G_RPmag": 7825.08},color2 = {"Jmag": 12375.60, "Hmag": 16476.02},)

In [ ]:
cluster_data = data_gaia[cluster & (data_gaia['probability'] >= 0.6)]
synthcl = asteca.synthetic(isochs)
my_cluster = asteca.cluster(
    cluster_df=cluster_data.to_pandas(),
    magnitude="Gmag",
    e_mag="e_Gmag",
    color="BP_RP",
    e_color='e_BP_RP',
    ra='ra',
    dec = 'dec',
    plx = 'parallax',
    pmra = 'pmra',
    pmde ='pmdec',
    color2 = 'RP_J',
    e_color2 = 'e_RP_J'
)
fix_params = {"alpha": 0.09, 'beta' : 0.94,"Rv": 3.1, "DR":0}
synthcl.calibrate(my_cluster,fix_params)

In [ ]:
import pymc as pm
import pytensor.tensor as pt
from pytensor.compile.ops import as_op

class SyntheticCluster(pt.Op):
    itypes = [pt.dscalar, pt.dscalar, pt.dscalar, pt.dscalar]
    otypes = [pt.dmatrix]

    def perform(self, node, inputs, outputs):
        met, loga, dm, Av = inputs
        params = {"met": met, "loga": loga, "dm": dm, "Av": Av}
        output = synthcl.generate(params)
        outputs[0][0] = np.column_stack((output[1], output[0],output[2]))
met_min, met_max, loga_min, loga_max = synthcl.min_max()
with pm.Model() as model:
    # Define priors
    dm = pm.Normal('dm', mu=10.3, sigma=0.2)
    loga = pm.Uniform('loga', lower=6, upper=7)
    Av = pm.Uniform('Av', lower=0.5, upper=2)
    met = pm.Uniform('met', lower=met_min, upper=met_max)
    # Use the custom Op to generate synthetic data
    synthetic_data = SyntheticCluster()(met, loga, dm, Av)

    observed_data = np.column_stack((cluster_data['BP_RP'], cluster_data['Gmag'], cluster_data['RP_J']))
    #weights = np.where((observed_data[1] <= 1*u.mag) & (observed_data[0] <= 14*u.mag) & (observed_data[2] <= 0.8*u.mag) ,80,1)
    # Likelihood: comparing synthetic to observed data
    sigma = pm.HalfNormal('sigma', sigma=10.0)
    likelihood = pm.Normal('likelihood', mu=synthetic_data, sigma=sigma, observed=observed_data)

In [ ]:
with model:
    trace = pm.sample(2000,tune=500,step = pm.DEMetropolisZ(tune='scaling', proposal_dist=pm.NormalProposal),chains=200,compute_convergence_checks=False)

In [ ]:
store_trace_results(trace,save_trace=True)

In [ ]:
results, trace_loaded = load_results(load_trace=True,only_last=True)

In [ ]:
#specific_datetime = '2024-04-30 20:23:33'
#selected_entry = results[results['Date_Time'] == specific_datetime]
selected_entry = results[results['Date_Time'] == results['Date_Time'].max()]

# Convert the last added rows back into dictionaries
fit_params_mean = selected_entry.set_index('Parameter')['Mean'].to_dict()
fit_params_median = selected_entry.set_index('Parameter')['Median'].to_dict()
fit_params_stds = selected_entry.set_index('Parameter')['Std'].to_dict()

# Print loaded dictionaries with the selected data
print(fit_params_mean)
print(fit_params_median)
print(fit_params_stds)

In [ ]:
az.summary(trace_loaded,var_names=['~likelihood','~likelihood_unobserved'])

In [ ]:
fig, ax = plt.subplots(4,4,layout='tight',figsize=(15,15))
az.plot_pair(trace_loaded,backend_kwargs = {'layout': 'tight'},kind='hexbin',var_names=['~likelihood','~likelihood_unobserved','~sigma']
             ,marginals=True,ax=ax,point_estimate='mode',textsize=18)
plt.show()
fig.savefig('../Tex_File/Figures/plot_pair_trace.pdf',bbox_inches='tight',dpi=600);

In [ ]:
fig_cmd, ax_cmd = plt.subplots(1,1, layout='tight', figsize=(7,7))
best_isochrone_median = synthcl.get_isochrone(fit_params_median)
best_isochrone_mean = synthcl.get_isochrone(fit_params_mean)
ax_cmd.scatter(data_gaia['BP_RP'][pms_high],data_gaia['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_cmd.scatter(data_gaia['BP_RP'][pms_low],data_gaia['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_cmd.scatter(data_gaia['BP_RP'][ms_high],data_gaia['Gmag'][ms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='MS Members',zorder=5)
ax_cmd.scatter(data_gaia['BP_RP'][ms_low],data_gaia['Gmag'][ms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='MS Probable Members',zorder=4)
ax_cmd.scatter(data_gaia['BP_RP'][high_tmass_nan],data_gaia['Gmag'][high_tmass_nan],s=80,alpha=0.95,lw=0,marker='*',c='k',label='Members',zorder=5)
ax_cmd.scatter(data_gaia['BP_RP'][low_tmass_nan],data_gaia['Gmag'][low_tmass_nan],s=35,alpha=0.85,lw=0,marker='d',c='k',label='Probable Members',zorder=4)
ax_cmd.plot(best_isochrone_mean[1],best_isochrone_mean[0],color='k',zorder=6,label=r'Best mean fit + $\mathcal{N}(\mu,\sigma)$',linestyle='--')
ax_cmd.plot(best_isochrone_median[1],best_isochrone_median[0],color='k',zorder=6,label='Best median fit',linestyle=':')
_, magbins = knuth_bin_width(data_gaia['Gmag'][cluster], return_bins=True)
_, colorbins = knuth_bin_width(data_gaia['BP_RP'][cluster], return_bins=True)
isochrones_masses = plot_hist2d(synthcl,fit_params_mean, fit_params_stds, ax_cmd,n_samples=1000,cmin=2000,alpha=0.3,bins=[colorbins,magbins],return_masses=True)
ax_cmd.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.tick_params(axis='both', which='both', direction='in')
ax_cmd.set(xlim=[-0.3,1.02*np.max(data_gaia['BP_RP'][cluster])],ylim=[0.99*np.min(data_gaia['Gmag'][cluster]),1.02*np.max(data_gaia['Gmag'][cluster])],adjustable='datalim')
ax_cmd.legend()
ax_cmd.set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(data_gaia['BP_RP'].unit),fontsize=16)
ax_cmd.set_ylabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit),fontsize=16)
#ax_cmd.annotate("", xy=(x0_gaia + dx_gaia, y0_gaia + dy_gaia), xytext=(x0_gaia, y0_gaia), arrowprops=prop)
plot_errors_bar(data_gaia[cluster]['Gmag'],data_gaia[cluster]['BP_RP'], data_gaia[cluster]['e_Gmag'], data_gaia[cluster]['e_BP_RP'], ax_cmd)
ax_cmd.invert_yaxis()
plt.show()
fig_cmd.savefig('../Tex_File/Figures/ngc6383_cmd.pdf',dpi=1500);

In [ ]:
fig_multicmd, ax_multicmd = plt.subplots(1,3, layout='tight', figsize=(15,7))
logAges = np.arange(6.2, 7.1, 0.2)  # From 6.1 to 7.0, inclusive
num_colors = len(logAges)
unique_color_indices = np.linspace(0, 1, num_colors, endpoint=False)
colors = [plt.cm.hsv(x) for x in unique_color_indices[::-1]]
ax_multicmd[0].scatter(data_gaia['BP_RP'][pms_high],data_gaia['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_multicmd[0].scatter(data_gaia['BP_RP'][pms_low],data_gaia['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_multicmd[0].scatter(data_gaia['BP_RP'][ms_high],data_gaia['Gmag'][ms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='MS Members',zorder=5)
ax_multicmd[0].scatter(data_gaia['BP_RP'][ms_low],data_gaia['Gmag'][ms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='MS Probable Members',zorder=4)
ax_multicmd[0].scatter(data_gaia['BP_RP'][high_tmass_nan],data_gaia['Gmag'][high_tmass_nan],s=80,alpha=0.95,lw=0,marker='*',c='k',label='Members',zorder=5)
ax_multicmd[0].scatter(data_gaia['BP_RP'][low_tmass_nan],data_gaia['Gmag'][low_tmass_nan],s=35,alpha=0.85,lw=0,marker='d',c='k',label='Probable Members',zorder=4)
ax_multicmd[0].set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(data_gaia['BP_RP'].unit),fontsize=16)
ax_multicmd[0].set_ylabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit),fontsize=16)
ax_multicmd[1].scatter(data_gaia['RP_J'][pms_high],data_gaia['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_multicmd[1].scatter(data_gaia['RP_J'][pms_low],data_gaia['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_multicmd[1].scatter(data_gaia['RP_J'][ms_high],data_gaia['Gmag'][ms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='MS Members',zorder=5)
ax_multicmd[1].scatter(data_gaia['RP_J'][ms_low],data_gaia['Gmag'][ms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='MS Probable Members',zorder=4)
ax_multicmd[1].set_xlabel(r'Color $(G_{{RP}} - J)$ [{}]'.format(data_gaia['BP_RP'].unit),fontsize=16)
ax_multicmd[1].set_ylabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit),fontsize=16)
#ax_multicmd[0].annotate("", xy=(x0 + dx, y0 + dy), xytext=(x0, y0), arrowprops=prop)

for i, logAge in enumerate(logAges):
    params = fit_params_mean.copy()
    params['loga'] = logAge
    isochrones_synth = synthcl.get_isochrone(params,average=False,return_masses=False)
    cut_isochrones = (isochrones_synth[0] >= np.nanmin(data_gaia['Gmag'].value))
    isochrones_synth = isochrones_synth[:,cut_isochrones]
    ax_multicmd[0].plot(isochrones_synth[1],isochrones_synth[0],lw=2,label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')
    ax_multicmd[1].plot(isochrones_synth[2],isochrones_synth[0],lw=2,label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')

ax_multicmd[0].scatter([],[],label=rf'$Z_{{ini}}={fit_params_mean['met']}$',s=0)
ax_multicmd[1].scatter([],[],label=rf'$Z_{{ini}}={fit_params_mean['met']}$',s=0)

# Swap x and y data in scatter plots
ax_multicmd[2].scatter(data_gaia['BP_RP'][pms_high], data_gaia['RP_J'][pms_high], s=80, alpha=0.95, lw=0, marker='*', c='red', label='PMS Members', zorder=5)
ax_multicmd[2].scatter(data_gaia['BP_RP'][pms_low], data_gaia['RP_J'][pms_low], s=35, alpha=0.85, lw=0, marker='d', c='orangered', label='PMS Probable Members', zorder=4)
ax_multicmd[2].scatter(data_gaia['BP_RP'][ms_high], data_gaia['RP_J'][ms_high], s=80, alpha=0.95, lw=0, marker='*', c='blue', label='MS Members', zorder=5)
ax_multicmd[2].scatter(data_gaia['BP_RP'][ms_low], data_gaia['RP_J'][ms_low], s=35, alpha=0.85, lw=0, marker='d', c='blueviolet', label='MS Probable Members', zorder=4)
# Update labels for swapped axes
ax_multicmd[2].set_xlabel(r'Color $(G_{{RP}} - G_{{RP}})$ [{}]'.format(data_gaia['BP_RP'].unit), fontsize=16)
ax_multicmd[2].set_ylabel(r'Color $(G_{{RP}} - J)$ [{}]'.format(data_gaia['BP_RP'].unit),fontsize=16)
# Swap x and y data in isochrones plot
for i, logAge in enumerate(logAges):
    params = fit_params_mean.copy()
    params['loga'] = logAge
    isochrones_synth = synthcl.get_isochrone(params)
    ax_multicmd[2].plot(isochrones_synth[1], isochrones_synth[2], lw=2, label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')
ax_multicmd[2].scatter([], [], label=rf'$Z_{{ini}}={fit_params_mean['met']}$', s=0)  # Placeholder for Zini label
ax_multicmd[0].set(xlim=[0.1*np.nanmin(data_gaia['BP_RP'][cluster]),1.02*np.nanmax(data_gaia['BP_RP'][cluster])],ylim=[0.95*np.nanmin(data_gaia['Gmag'][cluster]),1.02*np.max(data_gaia['Gmag'][cluster])])
ax_multicmd[1].set(xlim=[0.1*np.nanmin(data_gaia['RP_J'][cluster]),1.02*np.nanmax(data_gaia['RP_J'][cluster])],ylim=[0.95*np.nanmin(data_gaia['Gmag'][cluster]),1.02*np.max(data_gaia['Gmag'][cluster])])
for i, ax in enumerate(ax_multicmd.flatten()):
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(axis='both', which='both', direction='in') 
    ax.legend()
    if i != 2:
        ax.invert_yaxis()
fig_multicmd.savefig('../Tex_File/Figures/ngc6383_cmd_various.pdf',dpi=1500);

## Masses

In [ ]:
masses = pd.read_csv('masses_asteca.csv')

In [ ]:
masses['source_id'] = data_gaia[(data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.6)]['source_id']

In [ ]:
masses = QTable.from_pandas(masses)

In [ ]:
if not any(column in data_gaia.colnames for column in ['mass','mass_std']):
    data_gaia = join(data_gaia,masses,join_type='left',keys='source_id')
    cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.6)
    noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))
    data_gaia['mass'] = np.where(data_gaia['binar_prob'] >= 0.7, 
                                   data_gaia['m1'] + data_gaia['m2'], 
                                   data_gaia['m1'])

In [ ]:
for column_name in data_gaia.colnames:
    if hasattr(data_gaia[column_name], 'mask'):  # Check if the column has a mask attribute
        # Convert column to float if it's an integer type
        if np.issubdtype(data_gaia[column_name].dtype, np.integer):
            data_gaia[column_name] = data_gaia[column_name].astype(float)
        # Now safe to fill masked values with np.nan
        data_gaia[column_name] = data_gaia[column_name].filled(np.nan)

In [ ]:
np.nanmax(data_gaia['mass'])

In [ ]:
fig_massbinary, ax_massbinary = plt.subplots(2,1, figsize=(8,14))
best_isochrone_median = synthcl.get_isochrone(fit_params_median)
best_isochrone_mean = synthcl.get_isochrone(fit_params_mean)
sc_mass = ax_massbinary[0].scatter(data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['BP_RP'],data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['Gmag'],s=80,alpha=0.95,lw=0,marker='*',c=data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['mass'],label='Probable Members and Members',zorder=5,cmap='coolwarm_r')
sc_binary = ax_massbinary[1].scatter(data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['BP_RP'],data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['Gmag'],s=80,alpha=0.95,lw=0,marker='*',c=data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['binar_prob'],label='Probable Members and Members',zorder=5,cmap='coolwarm_r')
fig_massbinary.colorbar(sc_binary,location='right',pad=0.01).set_label('Binary probability',fontsize=16),fig_massbinary.colorbar(sc_mass,location='right',pad=0.01).set_label('Total mass',fontsize=16)
for ax in ax_massbinary.flatten():
    ax.plot(best_isochrone_mean[1],best_isochrone_mean[0],color='k',zorder=-1,label=r'Best mean fit',linestyle='--')
    ax.plot(best_isochrone_median[1],best_isochrone_median[0],color='k',zorder=6,label='Best median fit',linestyle=':')
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(axis='both', which='both', direction='in')
    ax.set(xlim=[-0.3,1.02*np.max(data_gaia['BP_RP'][cluster])],ylim=[0.99*np.min(data_gaia['Gmag'][cluster]),1.02*np.max(data_gaia['Gmag'][cluster])],adjustable='datalim')
    ax.legend()
    ax.set_ylabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit),fontsize=16)
    ax.invert_yaxis()
ax_massbinary[1].set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(data_gaia['BP_RP'].unit),fontsize=16)
fig_massbinary.subplots_adjust(hspace=0.01)
fig_massbinary.savefig('../Tex_File/Figures/ngc6383_mass_binary.pdf', bbox_inches='tight')
plt.show()

In [ ]:
prob_thresholds = [0.5, 0.6, 0.7, 0.8]

for i in prob_thresholds:
    condition = data_gaia[cluster]['probability'] >= i 
    condition_mag = (data_gaia[cluster]['probability'] >= i) & (data_gaia[cluster]['Gmag'] <= 18*u.mag)
    print(len(data_gaia[cluster][condition]),len(data_gaia[cluster][condition_mag]))

In [ ]:
def salpeter55(m):
    alpha = 2.35
    return m**-alpha


def millerscalo79(m):
    return np.where(m > 1, salpeter55(m), salpeter55(1))


def chabrier03individual(m):
    k = 0.158 * np.exp(-(-np.log10(0.08))**2 / (2 * 0.69**2))
    return np.where(m <= 1, 0.158 * (1. / m) * np.exp(-(
        np.log10(m) - np.log10(0.08))**2 / (2 * 0.69**2)), k * m**-2.3)


def chabrier03system(m):
    k = 0.086 * np.exp(-(-np.log10(0.22))**2 / (2 * 0.57**2))
    return np.where(m <= 1, 0.086 * (1. / m) * np.exp(
        -(np.log10(m) - np.log10(0.22))**2 / (2 * 0.57**2)), k * m**-2.3)


def kroupa01(m):
    """
    Kroupa (2001), 'On the variation of the initial mass function', Eq (2)
    """
    return np.where(
        m < 0.08, m**-0.3,
        np.where(m < 0.5, 0.08**-0.3 * (m / 0.08)**-1.3,
                 0.08**-0.3 * (0.5 / 0.08)**-1.3 * (m / 0.5)**-2.3))


def kroupa2002(x):
    """
    Kroupa (2002) piecewise Initial Mass Function (IMF) adapted for vectorized operations.
    Arguments:
    - x : array-like, masses for which the IMF is calculated

    Returns:
    - Array of IMF values corresponding to the input masses.
    """
    # Define the slopes and break points of the piecewise IMF
    alpha = np.array([-0.3, -1.3, -2.3])
    m0, m1, m2 = 0.01, 0.08, 0.5
    factors = np.array([(1. / m1) ** alpha[0], (1. / m1) ** alpha[1], ((m2 / m1) ** alpha[1]) * ((1. / m2) ** alpha[2])])

    # Initialize the output array with the same shape as input
    result = np.zeros_like(x, dtype=float)

    # Apply the piecewise function
    mask1 = (m0 < x) & (x <= m1)
    mask2 = (m1 < x) & (x <= m2)
    mask3 = (m2 < x)

    result[mask1] = factors[0] * (x[mask1] ** alpha[0])
    result[mask2] = factors[1] * (x[mask2] ** alpha[1])
    result[mask3] = factors[2] * (x[mask3] ** alpha[2])

    return result


# def kroupa1993(x):
#     """
#     Kroupa, Tout & Gilmore. (1993) piecewise IMF.
#     http://adsabs.harvard.edu/abs/1993MNRAS.262..545K
#     Eq. (13), p. 572 (28)
#     """
#     alpha = [-1.3, -2.2, -2.7]
#     m0, m1, m2 = [0.08, 0.5, 1.]
#     factor = [0.035, 0.019, 0.019]
#     if m0 < x <= m1:
#         i = 0
#     elif m1 < x <= m2:
#         i = 1
#     elif m2 < x:
#         i = 2
#     return factor[i] * (x ** alpha[i])


# def chabrier2001_log(x):
#     """
#     Chabrier (2001) lognormal form of the IMF.
#     http://adsabs.harvard.edu/abs/2001ApJ...554.1274C
#     Eq (7)
#     """
#     imf_vals = (1. / (np.log(10) * x)) * 0.141 * \
#         np.exp(-((np.log10(x) - np.log10(0.1)) ** 2) / (2 * 0.627 ** 2))

#     # Normalize PDF
#     int_imf = np.trapz(imf_vals, x)
#     imf_vals /= int_imf
#     return imf_vals


# def chabrier2001_exp(x):
#     """
#     Chabrier (2001) exponential form of the IMF.
#     http://adsabs.harvard.edu/abs/2001ApJ...554.1274C
#     Eq (8)
#     """
#     return 3. * x ** (-3.3) * np.exp(-(716.4 / x) ** 0.25)


# def salpeter1955(x):
#     """
#     Salpeter (1955)  IMF.
#     https://ui.adsabs.harvard.edu/abs/1955ApJ...121..161S/
#     """
#     return x ** -2.35

In [ ]:
def maxLkl(mass, alpha_min=-1, alpha_max=5):
    """
    Method defined in Khalaj & Baumgardt (2013):
    https://academic.oup.com/mnras/article/434/4/3236/960889
    and used in Sheikhi et al. (2016):
    https://academic.oup.com/mnras/article/457/1/1028/989829
    """

    def minfunc(alpha, x, xmin, xminmax, N):
        y = abs(
            alpha - (
                1 + N / (
                    np.sum(np.log(x / xmin))
                    - N * (np.log(xminmax) / (1 - xminmax**(alpha - 1)))
                )
            )
        )
        idx = np.argmin(y)
        return alpha[idx]

    # Slope on the original sample
    N = mass.size
    xmin, xmax = mass.min(), mass.max()
    xminmax = xmax / xmin
    if alpha_min <= 1:
        # Protect from -1 divergence
        alpha_r1 = list(np.linspace(alpha_min, .95, 2500))
        alpha_r2 = list(np.linspace(1.05, alpha_max, 2500))
        alpha_vals = np.array(alpha_r1 + alpha_r2)
    else:
        alpha_vals = np.linspace(alpha_min, alpha_max, 5000)
    alpha_lkl = minfunc(alpha_vals, mass, xmin, xminmax, N)

    N = mass.size
    alpha_std = (1/np.sqrt(N)) * (
        (alpha_lkl-1)**(-2) - np.log(xminmax)**2*(
            (xminmax**(alpha_lkl-1))/(1-xminmax**(alpha_lkl-1))**2))**(-.5)

    return alpha_lkl, alpha_std
def binnedIMF(mass, bins):
    """
    """
    # Histogram of all the masses
    yy, xx = ashist(mass, bins=bins, density=True)

    # Remove possible nans
    yy[np.isnan(yy)] = 0.
    # Align x values
    xx = .5 * (xx[1:] + xx[:-1])
    # Remove empty bins. Not sure if this has any impact
    msk = yy != 0
    yy, xx = yy[msk], xx[msk]

    # Perform LSF fit in logarithmic values
    alpha, intercept, alpha_std = logLSF(xx, yy)

    return bins, xx, yy, -alpha, intercept, alpha_std    
def logLSF(xx, yy):
    """
    """
    # Least squares fit
    y_log = np.log10(yy)
    # Mask -inf values that can appear when a bin contains 0 elements.
    msk = y_log == -np.inf
    x_log = np.log10(xx[~msk])
    y_log = y_log[~msk]
    pars, cov_matrix = np.polyfit(x_log, y_log, 1, cov=True)
    alpha, intercept = pars
    alpha_std = np.sqrt(cov_matrix[0][0])

    return alpha, intercept, alpha_std


In [ ]:
from matplotlib.ticker import LogFormatterSciNotation

def makeplot(mass_phot_msk, alpha_lkl, alpha_std, LSF_fits, alphas=None, alpha_stds=None, breakpoints=None):
    # Create a single figure and axis
    fig, ax = plt.subplots(figsize=(12, 8),layout='tight')

    # Calculate the range for the mass values

    if alphas:
        if breakpoints is None or len(breakpoints) == 0:
            breakpoints = [x0.max()]  # Only use the maximum value as a breakpoint to apply alpha across all x0
        breakpoints = np.concatenate(([x0.min()], breakpoints, [x0.max()]))
        # Plot piecewise power-law distributions for PyMC results
        for i, alpha in enumerate(alphas):
            # Define the segment of x values for this piece
            xs = x0[(x0 >= breakpoints[i]) & (x0 < breakpoints[i + 1])]
            y_vals_log_pymc = 10**intercept * xs**(-alpha)
            txt = r"$\alpha_{{pymc{0}}}={1:.3f}\pm{2:.3f}$".format(i+1, alpha, alpha_stds[i])
            ax.plot(xs, y_vals_log_pymc, '--', lw=2, label=txt, zorder=5)
            if i == 1:
                ax.axvline(breakpoints[i],c='k')

In [ ]:
def makeplot(mass_phot_msk, LSF_fits=None):
    fig, ax = plt.subplots(figsize=(12, 8),layout='tight')

    mass_data = mass_phot_msk[:, np.newaxis]
    # Define the bandwidths to test
    bandwidths = 10 ** np.linspace(-1, 10, 200)  # e.g., from 0.1 to 10
    
    # Set up the grid search with cross-validation
    grid = GridSearchCV(KernelDensity(kernel='gaussian'),
                    {'bandwidth': bandwidths},
                    cv=5,n_jobs=-1)  # 5-fold cross-validation
    
    # Perform the grid search on the data
    grid.fit(mass_data)
    kde = grid.best_estimator_
    support = np.linspace(mass_phot_msk.min(), mass_phot_msk.max(), 1000)[:, np.newaxis]
    densities = np.exp(kde.score_samples(support))
    # Plotting the results
    ax.plot(support, densities, 'k:', label="KDE of all masses")
    support = support.flatten()
    # Normalize at a specific mass point, e.g., 1 solar mass:
    normalization_mass = np.array([1.0])  # Ensure it's an array
    # Find closest index to this mass in support
    idx = np.abs(support - normalization_mass).argmin()
    y_norm = densities[idx]
    
    # Adjust the Kroupa 2002 normalization
    kroupa_scaled = y_norm * kroupa2002(support) / kroupa2002(normalization_mass)
    chabrier_scaled = y_norm * chabrier03system(support) / chabrier03system(normalization_mass)
    # Plotting Kroupa 2002 normalized
    ax.plot(support, kroupa_scaled, ls='-.', c='red', label='Kroupa 2002 normalized')
    ax.plot(support, chabrier_scaled, ls='-.', c='purple', label='Chabrier 2003 (system) normalized')
    min_yy = np.inf

    LSF_fits = []
    for bins in (5, 10, 20):
        LSF_fits.append(binnedIMF(mass_phot_msk, bins))
    for bins, xx, yy, alpha, intercept, alpha_std in LSF_fits:
        ax.scatter(xx, yy, s=100 - 2 * bins, marker='.', alpha=0.8,label=f"$\\alpha_{{LSF}}={alpha:.3f}\\pm{alpha_std:.3f} (N_{{\\mathrm{{bins}}}}={bins})$")
        min_yy = min(min_yy, min(yy))

    x0 = np.linspace(mass_phot_msk.min(), mass_phot_msk.max(), 100)
    intercept = LSF_fits[-1][4]  # Assuming LSF_fits[-1] is the most recent fit
    alpha_lkl, alpha_std = maxLkl(mass_phot_msk)
    # Plot likelihood results from traditional methods
    y_vals_log = 10**intercept * x0**(-alpha_lkl)
    txt = r"$\alpha_{{Lkl}}={:.3f}\pm{:.3f}$".format(alpha_lkl, alpha_std)
    ax.plot(x0, y_vals_log, 'k--', lw=2, label=txt, zorder=5)
    # Plot piecewise power-law distributions if alphas and breakpoints are provide
   
    ax.set_ylim(0.5*min_yy)
    ax.set_xlabel(r"$m\,[M_{\odot}]$",fontsize=16)
    ax.set_ylabel(r"$\xi(m) \Delta m$",fontsize=16)
    ax.set_yscale("log")
    ax.set_xscale("log")
    ax.legend(fontsize=11)
    return support, densities

In [ ]:
support, densities = makeplot(data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['m1']);

In [ ]:
with pm.Model() as model_new:
    log_k = pm.Uniform('log_k')
    k = pm.Deterministic('k',10**(log_k))
    sigma = pm.HalfNormal('sigma',sigma=10)
    likelihood = pm.Normal('likelihood',mu=np.log10(k*chabrier03system(support)),sigma=sigma,observed=np.log10(densities))
    trace = pm.sample(50000,tune=5000,nuts_sampler='numpyro',target_accept=0.95)

In [ ]:
az.summary(trace)

In [ ]:
fig,ax = plt.subplots(1,1,layout='tight')
ax.scatter(support,densities,s=5)

ax.scatter(support,1.084*chabrier03system(support),s=5)
#ax.set_yscale('log')
#ax.set_xscale('log')

## Parallax

In [ ]:
def parallax_determination(data, prob_thresholds=[50, 60, 70, 80], return_trace=False, progressbar=False, savefig=None, prior_type='uniform', paper_single=False, parallax_hist=True):
    prob_number = np.array(prob_thresholds) / 100
    num_types = 2 if paper_single else 3  # Adjust the number of types of graphics based on 'paper_single'
    
    # Adjust num_rows if parallax_hist is True
    num_rows = num_types + (1 if parallax_hist else 0)
    # Determine the grid layout
    num_cols = len(prob_number)  # Number of columns is equal to the number of thresholds
    
    # Create the subplots with dynamic layout
    if paper_single:
        figsize = (18,5.5)
    else:
        figsize = (6 * num_cols, 4.5 * num_rows)
    if paper_single:
        fig, axes = plt.subplots(num_cols,num_rows, figsize=figsize, constrained_layout=True)
    else:
        fig, axes = plt.subplots(num_rows, num_cols, figsize=figsize, constrained_layout=True)
    # Ensure axes is a 2D array for consistency in indexing
    axes = np.atleast_2d(axes)
    
    # If paper_single, adjust axes for a single plot type
    results_distance = {
         'mu_r_mean': [],
        'std_r_mean': [],
        'mu_r_std': [],
        'std_r_std': [],
    }
    results_parallax = {
        'mu_parallax_mean': [],
        'sigma_parallax_mean': [],
        'mu_parallax_std': [],
        'sigma_parallax_std': [],
    }
    all_traces = []
    min_data = (data['probability'] >= np.nanmin(prob_number))
    data = data[min_data]
    for j, prob_threshold in enumerate(prob_number):
        print(f"{prob_threshold * 100}%")
        # Filter data based on probability threshold
        filtered_data = data[data['probability'] >= prob_threshold]
        error_criteria = (filtered_data['parallax_error']/filtered_data['parallax']) <= 0.1
        useful_data = filtered_data[error_criteria]
        nonuseful_data = filtered_data[~error_criteria]
        # Assuming you have a way to get 'model_results' from your 'parallax_model' function call
        print('Distance sampling')
        distance_model_results = distance_model(useful_data, return_trace, progressbar)# Placeholder function call
        print('Parallax sampling')
        parallax_model_results = fit_parallax_model(useful_data, return_trace, progressbar,prior_distance=distance_model_results['mu_r_mean'],prior_type='uniform')
        mean_parallax = parallax_model_results['mu_parallax_mean']
        std_mean_parallax = parallax_model_results['mu_parallax_std']
        std_parallax = parallax_model_results['sigma_parallax_mean']
        if parallax_hist:
            ax0 = axes[0, j]
            ax0.axvline(mean_parallax, color='g', linestyle='--', label=r'$\mu_\varpi = {:.3f}\pm{:.3f}\,{}$'.format(mean_parallax,std_mean_parallax,u.mas))
            _,bins,_ = ax0.hist(useful_data['parallax'], bins='auto', color='red', alpha=0.95, histtype='step', label='Used parallax')
            ax0.hist(filtered_data['parallax'], bins=bins, color='orange', alpha=0.95, histtype='step', label='Parallax')
            ax0.hist(nonuseful_data['parallax'], bins=bins, color='green', alpha=0.95, histtype='step', label='Not used parallax')
            bin_width = np.diff(bins)[0]
            scaling_factor = len(useful_data) * bin_width
            x = np.linspace(mean_parallax - 3*std_parallax,mean_parallax + 3*std_parallax , 100)
            p = norm.pdf(x, mean_parallax, std_parallax) * scaling_factor  # Scale the PDF by the scaling factor
            ax0.plot(x, p)
            ax0.set_ylim(0, ax0.get_ylim()[1])
            ax0.fill_betweenx(y=[0, 1.5*ax0.get_ylim()[1]], x1=mean_parallax - std_mean_parallax, x2=mean_parallax + std_mean_parallax, color='blue', alpha=0.05,label='Standard desviation')
            ax0.set_xlabel(r'$\varpi$ [{}]'.format(data['parallax'].unit),fontsize=18)
            ax0.set_ylabel('Counts',fontsize=18)
            ax0.legend(loc='upper left')
            if paper_single:
                ax1 = axes[j,1]
            else:
                ax1 = axes[1, j]
        else:
            ax1 = axes[0, j]
        ax1.scatter(useful_data['parallax'], useful_data['Gmag'], s=10, alpha=0.9,c='red',edgecolor='blue',label='Used for distance estimation')
        ax1.scatter(nonuseful_data['parallax'], nonuseful_data['Gmag'], s=10, alpha=0.9,c='red',label='Not used for distance estimation')
        ax1.errorbar(filtered_data['parallax'], filtered_data['Gmag'], xerr=filtered_data['parallax_error'], fmt='none', alpha=0.3,capsize=0,elinewidth=0.5,color='gray')
        ax1.axvline(mean_parallax, color='g', linestyle='--', label=r'$\mu_\varpi = {:.3f}\pm{:.3f}\,{}$'.format(mean_parallax,std_mean_parallax,u.mas))
        ax1.set_xlabel(r'$\varpi$ [{}]'.format(data['parallax'].unit),fontsize=18)
        ax1.set_ylabel(r'$G_\mathrm{mag}$',fontsize=18)
        ax1.set_xlim(mean_parallax-15*std_parallax,mean_parallax+15*std_parallax)
        lim_y_ax1 = ax1.get_ylim()
        ax1.fill_betweenx(y=[lim_y_ax1[0], 1.5*lim_y_ax1[1]], x1=mean_parallax - std_parallax, x2=mean_parallax + std_parallax, color='blue', alpha=0.05,label='Standard desviation')
        ax1.set_ylim(lim_y_ax1)
        ax1.invert_yaxis()
        ax1.legend(framealpha=0.3,loc='upper left',fontsize=11)
        # Second type of graphic: Histogram of inverse parallax with sampled distance chains
        if parallax_hist:
            if paper_single:
                ax2 = axes[j,2]
            else:
                ax2 = axes[2, j]
        else:
            ax2 = axes[1, j]
        _,bins,_ = ax2.hist(1/useful_data['parallax'], bins='auto', color='red', alpha=0.95, histtype='step', label='Used inverse of parallax')
        ax2.hist(1 / filtered_data['parallax'], bins=bins, color='orange', alpha=0.95, histtype='step', label='Inverse of parallax')
        ax2.hist(1/nonuseful_data['parallax'], bins=bins, color='green', alpha=0.95, histtype='step', label='Not used inverse of parallax')
        # Store results
        for key in results_distance.keys():
            results_distance[key].append(distance_model_results[key])
        for key in results_parallax.keys():
            results_parallax[key].append(parallax_model_results[key])
        mean_distance = distance_model_results['mu_r_mean']
        std_distance = distance_model_results['std_r_mean']
        ax2.axvline(mean_distance, color='blue', linestyle='--',label=f'$\mu_d: {mean_distance:.3f} \pm$ {std_distance:.3f} kpc')
        ax2.set_ylim(0, ax2.get_ylim()[1])
        ax2.fill_betweenx(y=[0, 1.5*ax2.get_ylim()[1]], x1=mean_distance - std_distance, x2=mean_distance + std_distance, color='blue', alpha=0.05,label='Standard desviation')
        ax2.set_xlabel('Distance [kpc]',fontsize=18)
        ax2.set_ylabel('Counts',fontsize=18)
        ax2.legend(loc='upper left')
        
        # Fine-tuning the plots
        for ax in [ax1, ax2]:
            ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
            ax.tick_params(axis='both', which='both', direction='in')
        if return_trace:
            all_traces.append(distance_model_results['trace_distance'])
            all_traces.append(parallax_model_results['trace_parallax'])
    if savefig:
        if paper_single:
            plt.savefig(savefig + 'parallax_determination_paper.pdf',bbox_inches='tight')
        else:
            plt.savefig(savefig + 'parallax_determination.pdf',bbox_inches='tight')
    plt.show()
    results = {}
    results.update(results_distance),results.update(results_parallax)
    return results  # or any other results you wish to return

In [ ]:
parallax_results = parallax_determination(data_gaia[cluster], return_trace=True,savefig='../Tex_File/Figures/',prob_thresholds=[60],paper_single=True)

In [ ]:
parallax_results

## Proper motion

In [ ]:
def pm_determination(data, savefig=None, prob_number=[50, 60, 70, 80], progressbar=False,return_trace=False,return_pmdist=False,return_pmprob=False,paper_single=False):
    prob_number = np.array(prob_number) / 100
    if len(prob_number) == 1:
        fig, axes = plt.subplots(1, layout='constrained', figsize=(8,5))
    elif len(prob_number) == 2:
        fig, axes = plt.subplots(1, 2, layout='constrained', figsize=(17.5,7))
    elif len(prob_number) == 3:
        fig, axes = plt.subplots(1, 3, layout='constrained', figsize=(13,13))
    elif len(prob_number) == 4:
        fig, axes = plt.subplots(2, 2, layout='constrained', figsize=(13,13))
    else:
        raise ValueError("Unsupported number of probability thresholds")

    stats_results = []
    distance_pm = []
    pm_prob = []
    if len(prob_number) == 1:
        axes = np.array([axes])
    min_data = (data['probability'] >= np.nanmin(prob_number))
    data = data[min_data]
    # Determine the overall grid range
    total_pm_RA = data['pmra']
    total_pm_DEC = data['pmdec']
    added_space_ra = (np.nanmax(total_pm_RA) - np.nanmin(total_pm_RA)) / 8
    added_space_dec = (np.nanmax(total_pm_DEC) - np.nanmin(total_pm_DEC)) / 8
    grid_RA, grid_DEC = np.meshgrid(
        np.linspace(np.nanmin(total_pm_RA) - added_space_ra, np.nanmax(total_pm_RA) + added_space_ra, 700),
        np.linspace(np.nanmin(total_pm_DEC) - added_space_dec, np.nanmax(total_pm_DEC) + added_space_dec, 700)
    )

    for i, ax in zip(prob_number, axes.flatten()):
        print(f"{i * 100}%")
        # Filter the data based on the probability threshold
        probability_selection = (data['probability'] >= i)
        iterative_data = data[probability_selection]
        pm_RA = iterative_data['pmra']
        pm_DEC = iterative_data['pmdec']
        error_RA = iterative_data['pmra_error']
        error_Dec = iterative_data['pmdec_error']
        probability = iterative_data['probability']
        # Perform Bayesian analysis for the filtered data
        trace_results = FitProperMotion2DGaussian(pm_RA, pm_DEC, progressbar=progressbar,return_trace=return_trace)

        # Extract Bayesian analysis results
        bayesian_results = trace_results['results']
        
        # Extract and store the statistics
        stats = {
            'probability': i,
            'mu_RA_mean': bayesian_results['mu_RA_mean'],
            'mu_Dec_mean': bayesian_results['mu_Dec_mean'],
            'sigma_RA_mean': bayesian_results['sigma_RA_mean'],
            'sigma_Dec_mean': bayesian_results['sigma_Dec_mean'],
            'corr_mean': bayesian_results['corr_mean'],
            'mu_RA_std': bayesian_results['mu_RA_std'],
            'mu_Dec_std': bayesian_results['mu_Dec_std'],
            'sigma_RA_std': bayesian_results['sigma_RA_std'],
            'sigma_Dec_std': bayesian_results['sigma_Dec_std'],
            'corr_std': bayesian_results['corr_std']
        }
        if return_trace:
            stats['trace'] = trace_results.get('trace')
        stats_results.append(stats)
        # Create a 2D Gaussian distribution with mean parameters
        rv = multivariate_normal([bayesian_results['mu_RA_mean'], bayesian_results['mu_Dec_mean']],
                                 [[bayesian_results['sigma_RA_mean']**2, 
                                   bayesian_results['corr_mean'] * bayesian_results['sigma_RA_mean'] * bayesian_results['sigma_Dec_mean']],
                                  [bayesian_results['corr_mean'] * bayesian_results['sigma_RA_mean'] * bayesian_results['sigma_Dec_mean'], 
                                   bayesian_results['sigma_Dec_mean']**2]])
        density = rv.pdf(np.dstack([grid_RA, grid_DEC]))
        max_density_idx = np.argmax(density)
        max_density_coords = (grid_RA.ravel()[max_density_idx], grid_DEC.ravel()[max_density_idx])
            
        # Contour plot and scatter plot
        ax.contour(grid_RA, grid_DEC, density, levels=10, cmap=sns.color_palette("coolwarm", as_cmap=True))
        sc = ax.scatter(pm_RA, pm_DEC, s=25, alpha=0.8, marker='*',c=probability,cmap=sns.color_palette("coolwarm", as_cmap=True))
        fig.colorbar(sc,pad=0.01).set_label('Probability',fontsize=15)
        ax.scatter(bayesian_results['mu_RA_mean'], bayesian_results['mu_Dec_mean'], marker='1', s=400, color='blue')
        ax.axvline(bayesian_results['mu_RA_mean'], color='darkorange', ls='--')
        ax.axhline(bayesian_results['mu_Dec_mean'], color='darkorange', ls='--')
        # Axis labels and title
        ax.set_xlabel(r'$\mu_\alpha*$ [mas/yr]',fontsize=16)
        ax.set_ylabel(r'$\mu_\delta$ [mas/yr]',fontsize=16)

        # Fine-tuning the plot
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.tick_params(axis='both', which='both', direction='in')
        ax.set_aspect('equal')
        legend_label = r'$C = ({:.4f},{:.4f})\;{}$'.format(bayesian_results['mu_RA_mean'],bayesian_results['mu_Dec_mean'], (u.mas/u.yr))
        ax.legend([legend_label], loc='best', framealpha=0.8)
        if return_pmdist:
            distance_center_pm = np.sqrt((pm_RA - bayesian_results['mu_RA_mean']*(u.mas/u.yr))**2 + (pm_DEC - bayesian_results['mu_Dec_mean']*(u.mas/u.yr))**2)
            distancepm = {
                'probability': i,
                'distancepm': 1/distance_center_pm}
            distance_pm.append(distancepm)
            if return_pmprob:
                pmprob_ = (1/distance_center_pm)*iterative_data['probability']
                pm_prob.append(pmprob_)
    # Save the figure if a path is provided
    if savefig:
        if paper_single:
            fig.savefig(f"{savefig}proper_motion_2d_paper.pdf",bbox_inches='tight')
        else:
            fig.savefig(f"{savefig}proper_motion_2d.pdf",bbox_inches='tight')
    plt.show()
    if return_pmdist and return_pmprob:
        return stats_results, distance_pm, pm_prob
    elif return_pmdist:
        return stats_results, distance_pm
    else:
        return stats_results

In [ ]:
pm_results, pm_dist, pmprob = pm_determination(data_gaia[cluster],return_pmdist=True,return_pmprob=True,savefig='../Tex_File/Figures/',prob_number=[60],paper_single=True,return_trace=True)

In [ ]:
pm_results

## Projected velocity

In [ ]:
data_gaia['projected_velocity'] = np.sqrt(data_gaia['pmra']**2 + data_gaia['pmdec']**2)

In [ ]:
results_vp = velocity_determination(data_gaia[cluster], return_trace=True,savefig='../Tex_File/Figures/',prob_thresholds=[60],paper_single=True)

In [ ]:
results_vp

## Center determination

In [ ]:
centers, params_list = graph_center_determination(data=data_gaia[cluster],projection=wcs,weighted=True,weight_array=pmprob[0],weight_prob=60,prob_number=None
                                                  ,savefig='../Tex_File/Figures/',only_weight=True,paper_single=True,distance_scale=parallax_results['mu_r_mean'][0]*u.kpc)

In [ ]:
params_list

In [ ]:
data_gaia['d_center'] = angular_separation(data_gaia['ra'], data_gaia['dec'], centers[0][0][0], centers[0][0][1]).to(u.arcmin)

In [ ]:
centers

## Radial Density Profile (cite Andreas H. W. Kupper 2010)

3P King Porfile

$
f(R)  = k \left[\dfrac{1}{\sqrt{1+\left(\frac{R}{R_c}\right)^2}} - \dfrac{1}{\sqrt{1+\left(\frac{R_t}{R_c}\right)^2}}\right]^2+b
$

and 2P King Profile

$
f(R) = \dfrac{k}{1+\left(\frac{R}{R_c}\right)^2}+b
$

### Models

In [ ]:
hunt_mass = 902.27*u.Msun

In [ ]:
king_results = graph_king(data=data_gaia[cluster],centers=centers,savefig='../Tex_File/Figures/',
                          density_method='equip',distances=parallax_results['mu_r_mean'],
                          return_priors=True,prob_number=[0.6],paper_single=True,
                          cluster_mass=hunt_mass,log_scale=True)

In [ ]:
king_results

In [ ]:
seventh = (cluster) & (data_gaia['probability'] >= 0.6)
lf = data_gaia[seventh]
abs_mag = calculate_absolute_magnitude(lf['Gmag'],parallax_results['mu_r_mean']*u.kpc)

min_mag = np.floor(np.nanmin(abs_mag)).value
max_mag = np.ceil(np.nanmax((abs_mag))).value
fig,ax = plt.subplots(2,1,layout='tight',figsize=(7,10))
ax[0].hist(abs_mag,bins=np.arange(min_mag, max_mag + 1, 1),histtype='step',hatch='//',color='orange')
ax[0].set_xlabel(r'$G_\mathrm{abs}$ [mag]',fontsize=16)
ax[0].set_ylabel('Counts',fontsize=16)

min_mag = np.floor(np.min(lf['Gmag'])).value
max_mag = np.ceil(np.max(lf['Gmag'])).value

ax[1].hist(lf['Gmag'],bins=np.arange(min_mag, max_mag + 1, 1),histtype='step',hatch='//',color='orange')
ax[1].set_xlabel(r'$G_\mathrm{mag}$ [mag]',fontsize=16)
ax[1].set_ylabel('Counts',fontsize=16)

fig.savefig('../Tex_File/Figures/luminosity_function.pdf',bbox_inches='tight')

In [ ]:
from astropy.visualization.wcsaxes import SphericalCircle

In [ ]:
king_results['priors'][0]

In [ ]:
def half_mass_radius(data, centers, prob_number=70,distance=None):
    """
    Calculate the half-mass radius of a star cluster.

    Parameters:
    data (Table): Table of star data including 'mass' and 'probability'.
    center (tuple): Center of the cluster as a tuple (ra, dec).
    prob_number (float): Probability threshold to filter the data.

    Returns:
    float: The half-mass radius of the cluster in arcminutes.
    """
    # Filter data based on probability threshold
    centers = [SkyCoord(ra=centers[0], dec=centers[1], frame='icrs', unit='deg')]
    prob_threshold = prob_number / 100
    filtered_data = data[data['probability'] >= prob_threshold]
    # Calculate distances to the cluster center
    filtered_data['d_center'] = angular_separation(filtered_data['ra'], filtered_data['dec'], centers[0].ra, centers[0].dec)
    
    # Sort by distance from the center
    sorted_indices = np.argsort(filtered_data['d_center'])
    filtered_data = filtered_data[sorted_indices]

    # Calculate cumulative mass and find the half total mass
    cumulative_mass = np.cumsum(filtered_data['mass'])
    half_total_mass = np.max(cumulative_mass) / 2

    # Find the half-mass radius
    half_mass_index = np.searchsorted(cumulative_mass, half_total_mass)
    half_mass_radius = filtered_data['d_center'][half_mass_index]
    if distance:
        return half_mass_radius.to(u.arcmin),linear_size(half_mass_radius.to(u.arcmin),distance)
    else:
        return half_mass_radius.to(u.arcmin)

In [ ]:
R_t_mean =king_results['bayesian_results']['R_t_mean'][0]
R_c_mean =king_results['bayesian_results']['R_c_mean'][0]
R_h = king_results['bayesian_results']['half_light_radius']
R_hill = king_results['priors'][0]['hill_radius']
R_bound = king_results['priors'][0]['gravitational_bound_radius']
R_hm =half_mass_radius(data_gaia[cluster],centers=centers[0][0],distance=parallax_results['mu_parallax_mean'])

In [ ]:
R_h,R_hm

In [ ]:
# Create the figure and axes with the WCS projectio
fig_real, ax_real = plt.subplots(1, 1,figsize=(12,12), subplot_kw={'projection': wcs})

# Plot the FITS image
cc = ax_real.imshow(data, cmap='rainbow',aspect='equal',vmin=2500,vmax=16000)
ax_real.scatter(data_gaia[cluster]['ra'],data_gaia[cluster]['dec'],color='darkred', transform=ax_real.get_transform('world'),alpha=0.6,s=10,label='NGC 6383 sources',zorder=4)
ax_real.scatter(centers[0][0][0],centers[0][0][1],marker='1', s=400, color='blue', transform=ax_real.get_transform('world'),lw=1,alpha=0.85,label='Center')
from astropy.visualization.wcsaxes import SphericalCircle
import astropy.units as u

# Assuming 'ax_real' is already set up with an appropriate WCS projection
# Define colors for better differentiation
colors = ['red', 'darkslategrey', 'darkblue', 'darkred', 'magenta', 'yellow', 'orangered']
circle_search = SphericalCircle((263.67148711*u.deg, -32.57727207*u.deg), 40 * u.arcmin, edgecolor=colors[0], facecolor='none', transform=ax_real.get_transform('world'), label='Cone Search')
circle_search.set(linestyle='--', alpha=0.85, linewidth=1.6)
# Tidal radius circle
circle_tidal = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_t_mean, edgecolor=colors[1], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_t = {:.3f} \,{}$'.format(R_t_mean.value, u.arcmin))
circle_tidal.set(linestyle='--', alpha=0.85, linewidth=1.6)
# Core radius circle
circle_core = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_c_mean, edgecolor=colors[2], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_c = {:.3f}\,{}$'.format(R_c_mean.value, u.arcmin))
circle_core.set(linestyle='solid', alpha=0.85, linewidth=1.6)
# Half-light radius circle
circle_rh = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_h, edgecolor=colors[3], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hl}} = {:.3f}\,{}$'.format(R_h.value, u.arcmin))
circle_rh.set(linestyle='-.', alpha=0.85, linewidth=1.6)
# Half-mass radius circle
circle_rm = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_hm[0], edgecolor=colors[4], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hm}} = {:.3f}\,{}$'.format(R_hm[0].value, u.arcmin))
circle_rm.set(linestyle='-', alpha=0.85, linewidth=1.6)
# Hill radius circle
circle_hill = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_hill, edgecolor=colors[5], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hill}} = {:.3f}\,{}$'.format(R_hill.value, u.arcmin))
circle_hill.set(linestyle='dashdot', alpha=0.85, linewidth=1.6)
# Bound radius circle
circle_bound = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_bound, edgecolor=colors[6], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{bound}} = {:.3f}\,{}$'.format(R_bound.value, u.arcmin))
circle_bound.set(linestyle='solid', alpha=0.85, linewidth=1.6)

# Add all patches to the axes
ax_real.add_patch(circle_search),ax_real.add_patch(circle_tidal),ax_real.add_patch(circle_core),ax_real.add_patch(circle_rh),ax_real.add_patch(circle_rm),ax_real.add_patch(circle_hill),ax_real.add_patch(circle_bound)
ax_real.set_autoscale_on(False)

# Add coordinate axes
ax_real.set_xlabel(r'$\alpha$ [{}]'.format(data_gaia['ra'].unit),fontsize=16)
ax_real.set_ylabel(r'$\delta$ [{}]'.format(data_gaia['dec'].unit),fontsize=16)
ax_real.coords[0].set_major_formatter('d.dd')
ax_real.coords[1].set_major_formatter('d.dd')
ax_real.coords.grid(True, color='white', linestyle='dotted', alpha=0.5)
ax_real.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_real.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_real.tick_params(axis='both', which='both', direction='in',fontsize=14)
ax_real.set_aspect('equal')
gc_distance,scalebar_lenght = 1/(parallax_results['mu_parallax_mean'][0])*u.kpc,5 * u.pc
scalebar_angle = (scalebar_lenght / gc_distance).to(u.deg, equivalencies=u.dimensionless_angles())
add_scalebar(ax_real, scalebar_angle, label="5 pc", color="white",corner='bottom right');
ax_real.legend(loc='upper left')
fig_real.savefig('../Tex_File/Figures/real_sky.pdf',bbox_inches='tight',dpi=800);

In [ ]:
# Constants for the Hill and Tidal radii, assuming these are already defined elsewhere in your code
R_hill_value = R_hill  # in the same units as d_center, assuming arcmin here
R_tidal_value = R_t_mean # also assuming arcmin

# Filter the stars between the Hill and Tidal radii
between_hill_tidal = data_gaia[cluster][(data_gaia[cluster]['d_center'] >= R_hill_value) & (data_gaia[cluster]['d_center'] <= R_tidal_value)]

# Number of stars in the defined range
num_stars_between = len(between_hill_tidal)
print(f"Number of stars between Hill and Tidal radius: {num_stars_between}")

# Probabilities analysis
if 'probability' in between_hill_tidal.columns:
    lower_prob = between_hill_tidal['probability'].min()
    higher_prob = between_hill_tidal['probability'].max()
    mean_prob = between_hill_tidal['probability'].mean()
    print(f"Lower probability of sources: {lower_prob}")
    print(f"Higher probability of sources: {higher_prob}")
    print(f"Mean probability of sources: {mean_prob}")
else:
    print("Probability data is not available in the dataset.")


In [ ]:
def half_mass_relaxation_time(N, rh, M, lambda_value=0.02, G=G):
    """
    Calculate the half-mass relaxation time for a star cluster.
    
    Parameters:
    N (int): Number of cluster members.
    rh (float): Half-mass radius of the cluster in parsecs.
    M (float): Total mass of the cluster in solar masses.
    lambda_value (float): The constant from Giersz & Heggie (1994), typically around 0.02.
    G (float): Gravitational constant in units of pc (km/s)^2 / Msun.
    
    Returns:
    float: The half-mass relaxation time in millions of years.
    """
    # Calculate the half-mass relaxation time in Myr using the given formula
    t_rh = (0.17 * N) / (np.log(lambda_value * N)) * np.sqrt((rh**3) / (G * M))
    # Convert to Myr if needed (depends on units of G)
    return t_rh.to(u.Myr)
t_rh_hunt = half_mass_relaxation_time(N=len(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]), rh=R_hm[1], M=hunt_mass,lambda_value=0.2)
print(t_rh_hunt)

In [ ]:
np.nanmax(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['m1'])

In [ ]:
(t_rh_hunt/((10**(fit_params_median['loga'])*u.yr))).to(u.dimensionless_unscaled)

In [ ]:
data_gaia['m1'].unit = u.solMass

In [ ]:
t_seg = np.nanmean(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['m1'])/np.nanmax(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['m1'])*t_rh_hunt
print(t_seg)

In [ ]:
min_seg = np.nanmean(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['m1'])*(t_rh_hunt/((10**(fit_params_median['loga'])*u.yr))).decompose()
print(min_seg.to(u.solMass))

In [ ]:
np.nanmin(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['m1'])

### Cumulative plots

In [ ]:
extracted_centers = [(item[0][0], item[0][1]) for item in centers]
plot_cumulative(data=data_gaia[cluster],centers=extracted_centers,R_c=king_results['bayesian_results']['R_c_mean'],R_t=king_results['bayesian_results']['R_t_mean'] ,savefig='../Tex_File/Figures/')

In [ ]:
def plot_cumulative_by_brightness(data, centers, prob_number=[50, 60, 70, 80], brightness_ranges=None, savefig=None, R_c=None, R_t=None, normalize=False, ks=True,paper=False):
    prob_number = np.array(prob_number) / 100
    centers = [SkyCoord(ra=ra, dec=dec, frame='icrs', unit='deg') for ra, dec in centers]
    num_plots = len(prob_number)
    
    if R_c is None:
        R_c = [None] * len(prob_number)
    if R_t is None:
        R_t = [None] * len(prob_number)
    
    if brightness_ranges is None:
        gmag_sorted = np.sort(data['Gmag'])
        quartiles = np.percentile(gmag_sorted, [0, 25, 50, 75, 100])
        brightness_ranges = [(quartiles[i], quartiles[i+1]) for i in range(len(quartiles)-1)]
    
    figsize = (12,8) if len(prob_number) == 2 else (8,8)
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=figsize)
    axs = axs.flatten() if num_plots > 1 else [axs]
    
    all_distances = {brightness_range: [] for brightness_range in brightness_ranges}
    
    for i, ax, center, r_c, r_t in zip(prob_number, axs, centers, R_c, R_t):
        selected_data = data[data['probability'] >= i]
        selected_data['d_center'] = angular_separation(selected_data['ra'], selected_data['dec'], center.ra, center.dec).to(u.arcmin)
        x_radius = np.linspace(0, np.nanmax(selected_data['d_center'].value), 400)
        
        cumulative_counts = {brightness_range: [] for brightness_range in brightness_ranges}
        total_counts = {brightness_range: 0 for brightness_range in brightness_ranges}
        
        for (mag_min, mag_max) in brightness_ranges:
            bright_data = selected_data[(selected_data['Gmag'] > mag_min) & (selected_data['Gmag'] <= mag_max)]
            total_counts[(mag_min, mag_max)] = len(bright_data)
            all_distances[(mag_min, mag_max)].extend(bright_data['d_center'].value)
        
        for r in x_radius:
            for (mag_min, mag_max) in brightness_ranges:
                count = len(selected_data[(selected_data['d_center'] <= r*u.arcmin) & 
                                          (selected_data['Gmag'] > mag_min) & 
                                          (selected_data['Gmag'] <= mag_max)])
                cumulative_counts[(mag_min, mag_max)].append(count)
        
        for (mag_min, mag_max), counts in cumulative_counts.items():
            if total_counts[(mag_min, mag_max)] > 0:
                normalized_counts = np.array(counts) / total_counts[(mag_min, mag_max)] if normalize else counts
                label = rf'$G_{{mag}}$: {mag_min.value:.2f} to {mag_max:.2f}'
                ax.plot(x_radius, normalized_counts, label=label)
        
        if r_c is not None and r_c.value:
            ax.axvline(r_c.value, color='green', label=rf'$R_c = {r_c.value:.2f}$ {r_c.unit}', linestyle='-.', linewidth=2, alpha=0.8,zorder=-1)
        if r_t is not None and r_t.value:
            ax.axvline(r_t.value, color='red', label=rf'$R_t = {r_t.value:.2f}$ {r_t.unit}', linestyle='-.', linewidth=2, alpha=0.8,zorder=-1)
        
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.set_xlabel('Radius [arcmin]',fontsize=16)
        ax.set_ylabel('Normalized cumulative count' if normalize else 'Cumulative count',fontsize=16)
        ax.legend()
        if not paper:
            ax.set_title(f"{int(i * 100)}% probability members")
    
    if ks:
        print("K-S test results:")
        for ((min_i, max_i), dist_i) in all_distances.items():
            for ((min_j, max_j), dist_j) in all_distances.items():
                if (min_i, max_i) < (min_j, max_j):  # Avoid comparing the same range and ensure unique pairs
                    ks_stat, ks_pvalue = ks_2samp(dist_i, dist_j)
                    print(f"Between brightness range {min_i:.3f}, {max_i:.3f} and {min_j:.3f}, {max_j:.3f}: KS-statistic={ks_stat:.4f}, p-value={ks_pvalue:.4f}")

    if savefig and paper:
        plt.savefig(savefig + 'cumulative_by_brightness_paper.pdf', bbox_inches='tight')
    else:
        plt.savefig(savefig + 'cumulative_by_brightness.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
plot_cumulative_by_brightness(data=data_gaia[cluster],prob_number=[60],centers=extracted_centers,
                              R_c=king_results['bayesian_results']['R_c_mean'],R_t=king_results['bayesian_results']['R_t_mean'],
                              savefig='../Tex_File/Figures/',normalize=True,ks=True,paper=True)

In [ ]:
def plot_cumulative_by_mass_and_type(data, centers, prob_number=[70], savefig=None, normalize=False, ks=True,m_seg=None):
    prob_number = np.array(prob_number) / 100
    centers = [SkyCoord(ra=ra, dec=dec, frame='icrs', unit='deg') for ra, dec in centers]
    # Filter data based on probability
    data = data[data['probability'] >= prob_number[0]]
    if m_seg is not None:
        data = data[data['mass'] <= m_seg]
    # Remove NaN values and sort mass data
    mass_sorted = np.sort(data['mass'][~np.isnan(data['mass'])])
    quartiles = np.percentile(mass_sorted, [0, 25, 50, 75, 100])
    mass_ranges = [(quartiles[i], quartiles[i+1]) for i in range(4)]

    # Set up figure
    figsize = (18, 6)
    fig, axs = plt.subplots(1, 3, figsize=figsize, squeeze=False,sharex=True)
    axs = axs.flatten()


    # Calculate angular separation to cluster center
    data['d_center'] = angular_separation(data['ra'], data['dec'], centers[0].ra, centers[0].dec).to(u.arcmin)

    # Define the maximum radius for the x-axis
    max_radius = data['d_center'].max()

    # Split data into single and binary stars
    binary_data = data[data['binar_prob'] >= 0.6]
    single_data = data[data['binar_prob']  < 0.6]

    # Set up datasets for singles and binaries
    datasets = [('Single Stars', single_data), ('Binary Stars', binary_data)]
    ks_results = {}

    # Process each dataset
    for idx, (title, selected_data) in enumerate(datasets):
        ax = axs[idx]
        all_dists = []

        for (mass_min, mass_max) in mass_ranges:
            mask = (selected_data['mass'] >= mass_min) & (selected_data['mass'] < mass_max)
            dists = selected_data[mask]['d_center']
            # Skip the mass range if no data points are present
            if len(dists) == 0:
                continue
            all_dists.append(dists)
            # Compute cumulative distribution
            cumulative_dist = np.array([np.sum(dists <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
            if normalize:
                max_cumulative_dist = np.max(cumulative_dist)
                if max_cumulative_dist > 0:  # Protect against division by zero
                    cumulative_dist /= max_cumulative_dist
            label = fr'Mass: {mass_min.value:.2f} - {mass_max.value:.2f} $M_\odot$'
            ax.plot(np.linspace(0, max_radius, 400), cumulative_dist, label=label)
        ax.set_title(title,fontsize=16)
        ax.set_xlabel('Radius (arcmin)',fontsize=16)
        if idx == 0:
            ax.set_ylabel('Cumulative distribution' + (' (normalized)' if normalize else ''),fontsize=16)
        ax.legend()

        # Perform KS tests within each dataset if there's more than one mass range
        if ks and len(all_dists) > 1:
            for i in range(len(all_dists)):
                for j in range(i+1, len(all_dists)):
                    if len(all_dists[i]) > 0 and len(all_dists[j]) > 0:
                        ks_stat, ks_pvalue = ks_2samp(all_dists[i], all_dists[j])
                        ks_results[(title, f'Q{i+1} vs Q{j+1}')] = (ks_stat, ks_pvalue)

    # Perform KS test between single and binary stars if both have data
    if len(single_data['d_center']) > 0 and len(binary_data['d_center']) > 0:
        ks_stat, ks_pvalue = ks_2samp(single_data['d_center'], binary_data['d_center'])
        ks_results[('Single vs Binary Stars', 'Overall')] = (ks_stat, ks_pvalue)

    # Plot for combined single and binary stars
    ax = axs[2]
    single_cum_dist = np.array([np.sum(single_data['d_center'] <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
    binary_cum_dist = np.array([np.sum(binary_data['d_center'] <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
    if normalize:
        max_single = np.max(single_cum_dist)
        max_binary = np.max(binary_cum_dist)
        if max_single > 0:
            single_cum_dist /= max_single
        if max_binary > 0:
            binary_cum_dist /= max_binary
    ax.plot(np.linspace(0, max_radius, 400), single_cum_dist, label='Single Stars')
    ax.plot(np.linspace(0, max_radius, 400), binary_cum_dist, label='Binary Stars')
    ax.set_title('Single vs Binary Stars',fontsize=16)
    ax.set_xlabel('Radius (arcmin)',fontsize=16)
    ax.legend()

    # Setting minor ticks for the x-axis and y-axis
    for idx, ax in enumerate(axs):
        if idx != 0:
            ax.set_yticks([])
        if idx == 0:
            ax.yaxis.set_major_locator(ticker.AutoLocator())  # Auto-locate y-axis ticks
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto-locate minor ticks
        ax.xaxis.set_major_locator(ticker.AutoLocator())  # Auto-locate x-axis ticks
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto-locate minor tick
    plt.subplots_adjust(wspace=0, hspace=0)
    # Save the figure if a save path is provided
    if savefig:
        if m_seg is not None:
            plt.savefig(savefig + 'cumulative_by_mass_and_type_mseg.pdf', bbox_inches='tight')
        else:
            plt.savefig(savefig + 'cumulative_by_mass_and_type.pdf', bbox_inches='tight')
    plt.show()

    # Print KS test results
    print("K-S test results:")
    for key, (ks_stat, ks_pvalue) in ks_results.items():
        print(f"{key[0]} - {key[1]}: KS-stat={ks_stat:.4f}, p-value={ks_pvalue:.4f}")

In [ ]:
plot_cumulative_by_mass_and_type(data=data_gaia[cluster],prob_number=[60],centers=extracted_centers,savefig='../Tex_File/Figures/',normalize=True,ks=True)
#plot_cumulative_by_mass_and_type(data=data_gaia[cluster],prob_number=[70],centers=extracted_centers,R_c=king_results['bayesian_results']['R_c_mean'],savefig='../Tex_File/Figures/',normalize=True,ks=True)

In [ ]:
data_gaia[cluster]['m1'].max()

In [ ]:
data_gaia['mass'].unit = u.Msun 

In [ ]:
data_gaia[cluster & (data_gaia['probability'] >= 0.6)]['m1'].mean()

In [ ]:
data_gaia[cluster & (data_gaia['probability'] >= 0.6) & (data_gaia['m1'] <= min_seg)]['m1'].mean()

In [ ]:
min_seg

In [ ]:
plot_cumulative_by_mass_and_type(data=data_gaia[cluster],prob_number=[60]
                                 ,centers=extracted_centers,savefig='../Tex_File/Figures/'
                                 ,normalize=True,ks=True
                                 ,m_seg=min_seg.to(u.Msun))

In [ ]:
data_gaia['Q'] = data_gaia['J_H'] - 1.55 * data_gaia['H_K']

In [ ]:
np.sum(~np.isnan(data_gaia[cluster]['Q']))

In [ ]:
plt.hist(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q'],bins='auto')

In [ ]:
np.nanstd(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q']),np.nanmean(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q']),np.nanmedian(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q'])

In [ ]:
(len(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q'])-np.sqrt(len(data_gaia[cluster][data_gaia[cluster]['Q'] <= -0.05*u.mag]['Q'])))/len(data_gaia[cluster])

In [ ]:
#### Parallax observed vs Parallax corrected

fig_pp, ax_pp = plt.subplots(2,2, layout='tight',figsize=(12,8))

pp_sc_gmag = ax_pp[0,0].scatter(data_gaia['parallax_observed'],data_gaia['parallax'],1,c=data_gaia['Gmag'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[0,0].plot(ax_pp[0,0].get_xlim(), ax_pp[0,0].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_gmag,ax=ax_pp[0,0]).set_label(r'G magnitud [{}]'.format(data_gaia['Gmag'].unit))
ax_pp[0,0].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(data_gaia['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(data_gaia['Gmag'].unit))
ax_pp[0,0].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[0,0].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[0,0].tick_params(axis='both', which='both', direction='in')

pp_sc_nu = ax_pp[0,1].scatter(data_gaia['parallax_observed'],data_gaia['parallax'],1,c=data_gaia['nu_eff_used_in_astrometry'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[0,1].plot(ax_pp[0,1].get_xlim(), ax_pp[0,1].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_nu,ax=ax_pp[0,1]).set_label(r'$\nu_{{eff}}$ [{}]'.format((data_gaia['nu_eff_used_in_astrometry'].unit)))
ax_pp[0,1].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(data_gaia['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(data_gaia['nu_eff_used_in_astrometry'].unit))
ax_pp[0,1].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[0,1].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[0,1].tick_params(axis='both', which='both', direction='in')

pp_sc_ps = ax_pp[1,0].scatter(data_gaia['parallax_observed'],data_gaia['parallax'],1,c=data_gaia['pseudocolour'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[1,0].plot(ax_pp[1,0].get_xlim(), ax_pp[1,0].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_ps,ax=ax_pp[1,0]).set_label(r'Pseudolor [{}]'.format(data_gaia['pseudocolour'].unit))
ax_pp[1,0].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(data_gaia['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(data_gaia['pseudocolour'].unit))
ax_pp[1,0].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[1,0].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[1,0].tick_params(axis='both', which='both', direction='in')

pp_sc_lat = ax_pp[1,1].scatter(data_gaia['parallax_observed'],data_gaia['parallax'],1,c=data_gaia['ecl_lat'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[1,1].plot(ax_pp[1,1].get_xlim(), ax_pp[1,1].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_lat,ax=ax_pp[1,1]).set_label(r'Ecliptic Latitude [{}]'.format(data_gaia['ecl_lat'].unit))
ax_pp[1,1].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(data_gaia['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(data_gaia['ecl_lat'].unit))
ax_pp[1,1].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[1,1].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[1,1].tick_params(axis='both', which='both', direction='in')

fig_pp.suptitle('Corrected parallax and Observed parallax with different parameters')
fig_pp.savefig('../Tex_File/Figures/corr_obs_parallax.pdf');

### Histogram of the Angular distance between 2MASS and DR3

In [ ]:
### Angular distance

fig_cm, ax_cm = plt.subplots(1,1, layout='tight',figsize=(5,5),subplot_kw={'adjustable':'box'})
ax_cm.hist(data_gaia[cluster]['angular_distance'].data,bins='auto')
ax_cm.set(xlabel=r'Angular Distance GAIA - 2MASS [{}]'.format(data_gaia['angular_distance'].unit),ylabel=r'Number of sources')
ax_cm.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_cm.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cm.tick_params(axis='both', which='both', direction='in')
fig_cm.suptitle('Angular distance between GAIA DR3 and 2MASS')
fig_cm.savefig('../Tex_File/Figures/angular_separation.pdf');

## Radial velocities model

In [ ]:
plt.hist(data_gaia[cluster & (data_gaia['probability'] >= 0.7)]['radial_velocity'],bins='auto')

In [ ]:
np.nanmedian(data_gaia[cluster & (data_gaia['probability'] >= 0.6) & (data_gaia['binar_prob'] <= 0.6)]['radial_velocity'])

In [ ]:
np.nanstd(data_gaia[cluster & (data_gaia['probability'] >= 0.6) & (data_gaia['binar_prob'] <= 0.6)]['radial_velocity'])

In [ ]:
100*np.sum(~np.isnan(data_gaia[cluster & (data_gaia['probability'] >= 0.6) & (data_gaia['binar_prob'] <= 0.6)]['radial_velocity']))/len(data_gaia[cluster & (data_gaia['probability'] >= 0.6)])

In [ ]:
# Check for missing rv_amplitude_robust values
if hasattr(data_gaia[cluster]['rv_amplitude_robust'], 'mask'):
    condition_missing_amp = data_gaia[cluster]['rv_amplitude_robust'].mask
else:
    condition_missing_amp = np.isnan(data_gaia[cluster]['rv_amplitude_robust'])

# Create the plot
fig, ax = plt.subplots(layout='tight',figsize=(8,7))

# Group 1
ax.errorbar(data_gaia[cluster][condition_missing_amp]['radial_velocity'], data_gaia[cluster][condition_missing_amp]['Gmag'],
            xerr=data_gaia[cluster][condition_missing_amp]['radial_velocity_error'],
            fmt='o', ecolor='gray', mec='black', mfc='yellow', label='Sources without amplitude')

# Group 2
ax.errorbar(data_gaia[cluster][~condition_missing_amp]['radial_velocity'], data_gaia[cluster][~condition_missing_amp]['Gmag'],
            xerr=data_gaia[cluster][~condition_missing_amp]['radial_velocity_error'],
            fmt='none', ecolor='gray', mec='black', mfc='lightblue',zorder=-1)

scatter = ax.scatter(data_gaia[cluster][~condition_missing_amp]['radial_velocity'], 
                     data_gaia[cluster][~condition_missing_amp]['Gmag'],
                     c=data_gaia[cluster][~condition_missing_amp]['rv_amplitude_robust'].value,
                     s=60, edgecolor='black', cmap='coolwarm', label='Sources with amplitude')
# Color bar
cbar = fig.colorbar(scatter)
cbar.set_label(f'Amplitude [{data_gaia['rv_amplitude_robust'].unit}]',fontsize=16)


# Additional plot settings
ax.invert_yaxis()
ax.set_xlabel(f'Radial velocity [{data_gaia['radial_velocity'].unit}]',fontsize=16)
ax.set_ylabel(rf'$G_\mathrm{{mag}} $ [{data_gaia['Gmag'].unit}]',fontsize=16)
ax.legend()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.tick_params(axis='both', which='both', direction='in')
plt.show()
fig.savefig('../Tex_File/Figures/radial_velocity_amplitude.pdf',bbox_inches='tight')


In [ ]:
table_info = Simbad.list_columns('mesSpT')

In [ ]:
table_info

In [ ]:
from astroquery.simbad import Simbad

# Define the corrected query
query = """
SELECT main_id AS "Main Identifier", 
       basic.otype AS "Object Type", 
       ra AS "Right Ascension", 
       dec AS "Declination", 
       ids.ids AS "All Identifiers",
       h_link.membership AS "Membership Probability",
       cluster_table.id AS "Parent Cluster",
       otypedef.description AS "Object Type Definitions",
       mesSpT.sptype AS "Spectral Type"
FROM ident AS cluster_table
JOIN h_link ON cluster_table.oidref = h_link.parent
JOIN basic ON h_link.child = basic.oid
JOIN ids ON basic.oid = ids.oidref
LEFT JOIN otypedef ON basic.otype = otypedef.otype
LEFT JOIN mesSpT ON basic.oid = mesSpT.oidref
WHERE cluster_table.id = 'NGC 6383';
"""

# Execute the query using SIMBAD's TAP service
result_table = Simbad.query_tap(query)

# Print the results to check
print(result_table)

In [ ]:
# Function to find the preferred designation
def find_preferred_designation(all_ids, main_id):
    # Split the concatenated identifiers by '|'
    identifiers = all_ids.split('|')
    
    # Try to find identifiers in preferred order
    for prefix in ['Gaia DR3', 'Gaia', '2MASS']:
        for identifier in identifiers:
            if identifier.startswith(prefix):
                return identifier
    
    # If no preferred identifier is found, return the main identifier
    return main_id

# Apply the function to each row in the result_table
result_table['designation'] = [find_preferred_designation(row['All Identifiers'], row['Main Identifier']) for row in result_table]
#result_table['designation_dr2'] = [find_preferred_designation(row['All Identifiers'], row['Main Identifier']) for row in result_table]

In [ ]:
for i in result_table['designation','All Identifiers','Main Identifier']:
    if not i['designation'].startswith('Gaia DR3'):
        print(i['designation'])

In [ ]:
# Iterate through the table to update specific designation from Gaia DR2 to Gaia DR3
for i in range(len(result_table)):
    if 'Gaia DR2 4054565469499817728' in result_table['designation'][i]:
        # Replace 'Gaia DR2' with 'Gaia DR3'
        result_table['designation'][i] = result_table['designation'][i].replace('Gaia DR2', 'Gaia DR3')
        print(f"Updated row {i}: {result_table['designation'][i]}")


In [ ]:
result_table = unique(result_table,keys='designation')

In [ ]:
ct2020 = unique(QTable.read('../cantat_gaudin_2020.fit'),keys='GaiaDR2')
he2022 = unique(QTable.read('../he_2022.fit'),keys='GaiaEDR3')
jaehnig = unique(QTable.read('../Jaehnig_2021.fit'),keys='Gaia')

In [ ]:
from astropy.table import join, setdiff,unique

# Assuming 'he2022' and 'data_gaia' are your QTable objects and the columns 'GaiaEDR3' and 'source_id' contain the IDs

# Rename 'GaiaEDR3' in he2022 to 'source_id' to match data_gaia for joining
#he2022.rename_column('GaiaEDR3', 'source_id')

# Inner Join (common elements)
common_sources_he2022 = join(he2022, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_he2022 = setdiff(he2022, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_he2022 = setdiff(data_gaia[cluster], he2022, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_he2022))
print("Unique to HE2022:", len(unique_he2022))
print("Unique to Data Gaia:", len(unique_data_gaia_he2022))

In [ ]:
#aehnig.rename_column('Gaia', 'source_id')

common_sources_jaehnig = join(jaehnig, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_jaehnig = setdiff(jaehnig, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_jaehnig = setdiff(data_gaia[cluster], jaehnig, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_jaehnig))
print("Unique to Jaehnig:", len(unique_jaehnig))
print("Unique to Data Gaia:", len(unique_data_gaia_jaehnig))

In [ ]:
#ct2020.rename_column('GaiaDR2', 'source_id')

common_sources_ct = join(ct2020, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_ct = setdiff(ct2020, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_ct = setdiff(data_gaia[cluster], ct2020, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_ct))
print("Unique to CT:", len(unique_ct))
print("Unique to Data Gaia:", len(unique_data_gaia_ct))

In [ ]:
np.unique(result_table['Object Type Definitions'])

In [ ]:
data_gaia['source_id'][0]

In [ ]:
len('4054177651131258240')

In [ ]:
len('405456286245800')

In [ ]:
hunt2024 = QTable.read('/Users/notluquis/Downloads/members.csv')['source_id','mass_50']

In [ ]:
hunt2024

In [ ]:
common_sources_hunt2024 = join(hunt2024, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_hunt2024 = setdiff(hunt2024, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_hunt2024 = setdiff(data_gaia[cluster], hunt2024, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_hunt2024))
print("Unique to CT:", len(unique_hunt2024))
print("Unique to Data Gaia:", len(unique_data_gaia_hunt2024))

In [ ]:
common_sources_simbad = join(result_table, data_gaia[cluster], keys='designation', join_type='inner')

# Set Difference (elements unique to he2022)
unique_simbad = setdiff(result_table, data_gaia[cluster], keys='designation')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_simbad = setdiff(data_gaia[cluster], result_table, keys='designation')

# Print results
print("Common Sources:", len(common_sources_simbad))
print("Unique to CT:", len(unique_simbad))
print("Unique to Data Gaia:", len(unique_data_gaia_simbad))

In [ ]:
np.unique(common_sources_simbad['Object Type Definitions'])

In [ ]:
a = np.unique(common_sources_simbad['Spectral Type'])

In [ ]:
result_table[result_table['Object Type Definitions'] == 'T Tauri Star']

In [ ]:
common_sources_simbad[common_sources_simbad['Object Type Definitions'] == 'T Tauri Star']['designation','Spectral Type','probability','binar_prob','mass','pms_sagitta','av_sagitta']

In [ ]:
common_sources_simbad[common_sources_simbad['Spectral Type'] == a[-1]]

In [ ]:
common_sources_simbad[common_sources_simbad['Spectral Type'] == a[3]]

In [ ]:
common_sources_simbad[common_sources_simbad['Object Type Definitions'] == 'Eclipsing Binary']

## Other stuff 

In [ ]:
def cmd(data,cluster_element,isocrone_df=[0],z_array=[0]*4,modulus_distance=0,absorption=0,age_array=[0]*4,prob_number=[50,60,70,80], save_path='../Tex_File/Figures/HR_ngc6383.pdf',id=0):
    prob_number = np.array(prob_number) / 100
    num_plots = len(prob_number)
    # Setup subplots
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=(13, 9*num_plots))
    axs = axs.flatten() if num_plots > 1 else [axs]
    data_cluster = data[cluster_element]
    for i, ax,z,a in zip(prob_number, axs,z_array,age_array):
        # Filter using query or boolean indexing
        filtered_isochrone = isocrone_df.query(f"logAge == {a} and Zini == {z}")
        probability_th = (data_cluster['probability'] >= i)
        selected_data = data_cluster[probability_th]
        ax.scatter(data['BP_RP'][~cluster_element],data['Gmag'][~cluster_element],alpha=0.05,c='gray')
        ax.scatter(data_cluster['BP_RP'][~probability_th],data_cluster['Gmag'][~probability_th],alpha=0.05,c='gray',edgecolors='black')
        sc_cmd = ax.scatter(selected_data['BP_RP'],selected_data['Gmag'],c=selected_data['probability'],cmap=sns.color_palette("flare", as_cmap=True),s=14)
        fig.colorbar(sc_cmd).set_label(r'Probabilities')
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.tick_params(axis='both', which='both', direction='in')
        ax.set(xlabel=(r'Color $(BP - RP)$ [{}]'.format(data['BP_RP'].unit)),ylabel=(r'Magnitude $G$ [{}]'.format(data['Gmag'].unit)))
        x,y = ax.get_xlim(),ax.get_ylim()
        ax.scatter(filtered_isochrone['BP_RP']+absorption, filtered_isochrone['Gmag']+modulus_distance, s=5, c='blue', label=f'Age: 10^{a} years, Z={z}')
        ax.set(ylim=y,xlim=x)
        ax.invert_yaxis()
    fig.savefig('../Tex_File/Figures/HR_ngc6383.pdf');
cmd(data_gaia,cluster,isocrone_df=all_isochrones_DR3,prob_number=[70])#,z_array=[0.0201]*4,age_array=[6.5]*4,modulus_distance=10.47,absorption=0.156)

def cmd(data, cluster_element, isochrone_df, modulus_distance=0, absorption=0, prob_number=[50,60,70,80], save_path='../Tex_File/Figures/HR_ngc6383.pdf', id=None, z_array=None, age_array=None):
    prob_number = np.array(prob_number) / 100
    num_plots = len(prob_number)
    # Setup subplots
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=(13, 7 * num_plots // 2.5))
    axs = axs.flatten() if num_plots > 1 else [axs]
    data_cluster = data[cluster_element]

    # If an ID is provided, filter using ID, ignoring z_array and age_array
    if id is not None:
        filtered_isochrone = isochrone_df[isochrone_df['isochrone_id'] == id]
        # Plot for a single isochrone ID
        plot_isochrone_data(axs, data, cluster_element, filtered_isochrone, modulus_distance, absorption, prob_number)
    else:
        # Proceed with z_array and age_array if no ID is provided
        if z_array is not None and age_array is not None:
            for i, ax, z, a in zip(prob_number, axs, z_array, age_array):
                filtered_isochrone = isochrone_df.query(f"logAge == {a} and Zini == {z}")
                plot_isochrone_data(ax, data, cluster_element, filtered_isochrone, modulus_distance, absorption, [i])
        else:
            raise ValueError("Either an ID or both z_array and age_array must be provided.")

    fig.savefig(save_path)

def plot_isochrone_data(axs, data, cluster_element, filtered_isochrone, modulus_distance, absorption, prob_numbers):
    for i, ax in zip(prob_numbers, axs):
        probability_th = data[cluster_element]['probability'] >= i
        selected_data = data[cluster_element][probability_th]
        ax.scatter(data['BP_RP'][~cluster_element], data['Gmag'][~cluster_element], alpha=0.05, c='gray')
        ax.scatter(selected_data['BP_RP'], selected_data['Gmag'], c=selected_data['probability'], cmap=sns.color_palette("flare", as_cmap=True), s=14, label=f'Prob >= {i*100}%')
        if not filtered_isochrone.empty:
                ax.scatter(filtered_isochrone['BP_RP'] + absorption, filtered_isochrone['Gmag'] + modulus_distance, s=5, c='blue', label='Isochrone')
        ax.legend()
        ax.invert_yaxis()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(axis='both', which='both', direction='in')
        ax.set_xlabel('Color $(BP - RP)$')
        ax.set_ylabel('Magnitude $G$')

# ASTECA

In [ ]:
# ### =============================================================================
# ### #                                   ASTECA formatting.
# #### =============================================================================

write_data = data_gaia[cluster & (data_gaia['probability'] >= 0.6) & (data_gaia['Gmag'] <= 18*u.mag)]
pms = (write_data['pms_sagitta'] >= 0.6)
nopms = (write_data['pms_sagitta'] < 0.6)
filedir = '../ASteCA/input/'
#for i in ['2mass','dr3']:
for i in ['dr3']:
    if i == '2mass':
        write = write_data['designation','ra','dec','parallax','parallax_error','pmra','pmra_error','pmdec','pmdec_error','j_m','RP_J','j_msigcom','e_BP_RP']
        write.rename_columns(['j_m','j_msigcom'],['Jmag','e_Jmag'])
        ascii.write(write,f'{filedir}NGC_6383_{i}_all_18mag.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[pms],f'{filedir}NGC_6383_{i}_pms.csv',format='csv',overwrite=True,delimiter=',')
        #scii.write(write[nopms],f'{filedir}NGC_6383_{i}_nopms.csv',format='csv',overwrite=True,delimiter=',')
    if i == "dr3":
        write = write_data['designation','ra','dec','parallax','parallax_error','pmra','pmra_error','pmdec','pmdec_error','Gmag','BP_RP','e_Gmag','e_BP_RP']
        print(len(write))
        ascii.write(write,f'{filedir}NGC_6383_{i}_all_18mag.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[pms],f'{filedir}NGC_6383_{i}_pms.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[nopms],f'{filedir}NGC_6383_{i}_nopms.csv',format='csv',overwrite=True,delimiter=',')

## Outlier score

In [ ]:
import pandas as pd

In [ ]:
hunt24 = pd.read_csv('/Users/notluquis/Downloads/gogo/members.csv')

In [ ]:
ngc6383 = hunt24[hunt24['name'] == "NGC_6383"]

In [ ]:
ngc6383

In [ ]:
masses = ngc6383['mass_50']
masses = masses[masses > 0]
n, bins = np.histogram(masses, np.arange(np.min(masses), np.max(masses), 0.5))

# Calcula el centro de cada bin
M = 0.5 * (bins[1:] + bins[:-1])
dM = np.diff(bins)
dN = n

# Para dN/dM, divide el número de objetos por el ancho del bin
dN_dM = dN / dM

# Calcula los errores de Poisson como la raíz cuadrada de los conteos en cada bin
errors = np.sqrt(n) / dM

fig, ax = plt.subplots(1, 1, layout='tight', figsize=(7, 3))

# Crear el gráfico con barras de error
ax.errorbar(M, dN_dM, yerr=errors, marker='o', linestyle='None', capsize=5)

# Añadir títulos y etiquetas
ax.set_xlabel('Masa (M)')
ax.set_ylabel('dN/dM')
ax.set_yscale('log')
ax.set_xscale('log')
plt.show()

In [ ]:
cluster_criteria = hunt24['name'] == 'NGC_6383'
ngc6383 = hunt24[cluster_criteria]

In [ ]:
ngc6383['mass_2.5'].sum()

In [ ]:
ngc6383['mass_16'].sum()

In [ ]:
ngc6383['mass_50'].sum()

In [ ]:
ngc6383['mass_84'].sum()

In [ ]:
ngc6383['mass_97.5'].sum()

In [ ]:
gg = pd.read_csv('/Users/notluquis/Downloads/clusters.csv')

In [ ]:
pp = gg[gg['name'] == 'NGC_6383']

In [ ]:
pd.set_option('display.max_columns', None)